# Detección de cumplimiento de EPP con YOLOv8 — Ejecución en Colab

Reproduce el proyecto en Google Colab a partir del repositorio. Dos modos:

- **Modo A — Pipeline completo:** descarga del dataset, entrenamiento, evaluación en test
  y demostración del verificador (`run_all.sh`). Requiere una API key de Roboflow.
- **Modo B — Inferencia con el modelo entrenado:** ejecuta el verificador sobre imágenes
  o video sin reentrenar (requiere `best.pt`).

Requisito: activar GPU en `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`.


## Entorno

In [ ]:
!nvidia-smi

Clonar el repositorio (ajustar `REPO_URL` si corresponde):

In [ ]:
REPO_URL = "https://github.com/lguisadom/deteccion-epp-yolo.git"  #@param {type:"string"}
!git clone $REPO_URL
%cd deteccion-epp-yolo


Instalar dependencias (se reutiliza el `torch`+CUDA preinstalado en Colab):

In [ ]:
!pip install -q ultralytics roboflow python-dotenv
import torch; print("Torch", torch.__version__, "| CUDA:", torch.cuda.is_available())


## Modo A — Pipeline completo

Configurar la API key de Roboflow (app.roboflow.com → Settings → API → Private API Key):


In [ ]:
ROBOFLOW_API_KEY = ""  #@param {type:"string"}
open(".env","w").write(f"ROBOFLOW_API_KEY={ROBOFLOW_API_KEY}\n")


Ejecutar el pipeline de extremo a extremo con un único comando:

In [ ]:
!bash run_all.sh

Visualizar resultados generados:

In [ ]:
import os, glob
from IPython.display import Image, display
for p in ["outputs/training/yolov8s_baseline/results.png",
          "outputs/training/yolov8s_baseline/confusion_matrix.png",
          "report/figures/demo_cumplimiento.png"]:
    if os.path.exists(p): display(Image(p))
for p in sorted(glob.glob("outputs/compliance/images/*.jpg"))[:3]:
    display(Image(p))


## Modo B — Inferencia con el modelo entrenado

`best.pt` no se versiona en el repositorio (tamaño). Obtenerlo desde Google Drive o desde
un Release de GitHub (descomentar una opción):


In [ ]:
import os
os.makedirs("outputs/training/yolov8s_baseline/weights", exist_ok=True)
# Opción 1 — Google Drive:
# from google.colab import drive; drive.mount('/content/drive')
# !cp "/content/drive/MyDrive/EPP/best.pt" outputs/training/yolov8s_baseline/weights/best.pt
# Opción 2 — Release de GitHub:
# !wget -O outputs/training/yolov8s_baseline/weights/best.pt "https://github.com/<usuario>/deteccion-epp-yolo/releases/download/v1.0/best.pt"
print("best.pt:", os.path.exists("outputs/training/yolov8s_baseline/weights/best.pt"))


Subir imágenes (`.jpg`/`.png`) o un video (`.mp4`) y ejecutar el verificador:

In [ ]:
import os, sys, subprocess, cv2
import matplotlib.pyplot as plt
from google.colab import files
sys.path.insert(0, "src")
from ultralytics import YOLO
from compliance_checker import PPEComplianceChecker, Detection, annotate_frame

WEIGHTS = "outputs/training/yolov8s_baseline/weights/best.pt"
assert os.path.exists(WEIGHTS), "Falta best.pt (ver celda anterior o ejecutar el Modo A)."
model = YOLO(WEIGHTS); checker = PPEComplianceChecker(min_conf=0.25)

for fn in files.upload():
    if fn.lower().endswith((".jpg",".jpeg",".png")):
        r = model(fn, conf=0.25, verbose=False)[0]
        dets = [Detection(model.names[int(b.cls)], float(b.conf),
                tuple(float(v) for v in b.xyxy[0].tolist())) for b in r.boxes]
        st, rate = checker.check_frame(dets)
        ann = annotate_frame(cv2.imread(fn), st, rate)
        plt.figure(figsize=(12,8)); plt.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
        plt.axis("off"); plt.title(fn); plt.show()
    elif fn.lower().endswith((".mp4",".avi",".mov")):
        os.makedirs("data/videos", exist_ok=True); os.replace(fn, f"data/videos/{fn}")
        subprocess.run([sys.executable, "src/09_predict_video.py", f"data/videos/{fn}"], check=True)
